In [ ]:
%pip install requests tqdm pillow

In [ ]:
import os
import csv
import requests
from tqdm import tqdm
from PIL import Image
from io import BytesIO

# ---------------------------
# PARAMETRES
# ---------------------------

DATASET_DIR = "dataset_aromatic_plants"
CSV_FILE = "dataset_aromatic_plants/dataset.csv"

IMAGES_PER_PLANT = 30
IMAGE_SIZE = (400, 400)

plants = {
    "basil": "Ocimum basilicum",
    "mint": "Mentha",
    "rosemary": "Salvia rosmarinus",
    "thyme": "Thymus vulgaris",
    "oregano": "Origanum vulgare",
    "parsley": "Petroselinum crispum",
    "chives": "Allium schoenoprasum",
    "sage": "Salvia officinalis",
    "lavender": "Lavandula",
    "dill": "Anethum graveolens",
    "tarragon": "Artemisia dracunculus",
    "coriander": "Coriandrum sativum",
    "lemongrass": "Cymbopogon citratus",
    "bay_leaf": "Laurus nobilis",
    "marjoram": "Origanum majorana",
    "savory": "Satureja hortensis",
    "fennel": "Foeniculum vulgare",
    "anise": "Pimpinella anisum",
    "stevia": "Stevia rebaudiana",
    "lemon_balm": "Melissa officinalis",
    "chamomile": "Matricaria chamomilla",
    "pandan": "Pandanus amaryllifolius",
    "holy_basil": "Ocimum tenuiflorum",
    "lemon_verbena": "Aloysia citrodora",
    "wintergreen": "Gaultheria procumbens",
    "mugwort": "Artemisia vulgaris",
    "hyssop": "Hyssopus officinalis",
    "angelica": "Angelica archangelica",
    "lovage": "Levisticum officinale",
    "borage": "Borago officinalis"
}

# ---------------------------
# CREATION DOSSIERS
# ---------------------------

os.makedirs(DATASET_DIR, exist_ok=True)

csv_rows = []

# ---------------------------
# TELECHARGEMENT + RESIZE
# ---------------------------

def download_and_resize(url, filepath):

    try:
        r = requests.get(url, timeout=15)

        img = Image.open(BytesIO(r.content)).convert("RGB")

        img = img.resize(IMAGE_SIZE)

        img.save(filepath, "JPEG", quality=90)

        return True

    except:
        return False


# ---------------------------
# TELECHARGEMENT DATASET
# ---------------------------

for plant_name, taxon in plants.items():

    print(f"\nTéléchargement : {plant_name}")

    plant_dir = os.path.join(DATASET_DIR, plant_name)

    os.makedirs(plant_dir, exist_ok=True)

    count = 0
    page = 1

    pbar = tqdm(total=IMAGES_PER_PLANT)

    while count < IMAGES_PER_PLANT:

        url = "https://api.inaturalist.org/v1/observations"

        params = {
            "taxon_name": taxon,
            "quality_grade": "casual",
            "identified": "true",
            "photos": "true",
            "per_page": 200,
            "page": page
        }

        response = requests.get(url, params=params).json()

        results = response["results"]

        if not results:
            break

        for obs in results:

            for photo in obs["photos"]:

                img_url = photo["url"].replace("square", "large")

                filename = f"{count}.jpg"
                filepath = os.path.join(plant_dir, filename)

                success = download_and_resize(img_url, filepath)

                if success:

                    relative_path = os.path.join(plant_name, filename)

                    csv_rows.append([plant_name, relative_path])

                    count += 1
                    pbar.update(1)

                if count >= IMAGES_PER_PLANT:
                    break

            if count >= IMAGES_PER_PLANT:
                break

        page += 1

    pbar.close()


# ---------------------------
# ECRITURE CSV
# ---------------------------

with open(CSV_FILE, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow(["plant_type", "image_path"])

    writer.writerows(csv_rows)

print("\nDataset et CSV créés avec succès.")

In [ ]:
import requests
import os
from PIL import Image
from io import BytesIO
from tqdm import tqdm

ACCESS_KEY = "YOUR_UNSPLASH_API_KEY"

plants = [
    "basil plant",
    "mint plant",
    "thyme plant",
    "rosemary plant",
    "oregano plant"
]

DATASET_DIR = "dataset_plants"
IMAGES_PER_PLANT = 100
IMAGE_SIZE = (224,224)

os.makedirs(DATASET_DIR, exist_ok=True)

for plant in plants:

    plant_name = plant.replace(" ", "_")
    folder = os.path.join(DATASET_DIR, plant_name)

    os.makedirs(folder, exist_ok=True)

    page = 1
    count = 0

    pbar = tqdm(total=IMAGES_PER_PLANT)

    while count < IMAGES_PER_PLANT:

        url = "https://api.unsplash.com/search/photos"

        params = {
            "query": plant,
            "page": page,
            "per_page": 30,
            "client_id": ACCESS_KEY
        }

        r = requests.get(url, params=params).json()
        print(r)
        for photo in r["results"]:

            img_url = photo["urls"]["regular"]

            try:
                img = Image.open(BytesIO(requests.get(img_url).content)).convert("RGB")

                img = img.resize(IMAGE_SIZE)

                img.save(f"{folder}/{count}.jpg")

                count += 1
                pbar.update(1)

            except:
                pass

            if count >= IMAGES_PER_PLANT:
                break

        page += 1

    pbar.close()

In [ ]:
"""
Téléchargement en masse — 30 aromates communs via l'API iNaturalist
====================================================================
- Dossiers nommés par nom commun français
- Photos de FEUILLES prioritaires (annotation iNaturalist term_id=12/13)
- Fallback automatique sur toutes photos si quota non atteint

Arborescence produite :
  images/
    basilic_commun/
      basilic_commun_0001.jpg
      …
    thym_commun/
      …
    dataset.csv
"""

import os
import csv
import time
import requests
from PIL import Image
from io import BytesIO

# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────
TARGET_COUNT  = 50
IMAGE_SIZE    = (400, 400)
OUTPUT_DIR    = "dataset_aromatic"
CSV_FILENAME  = "dataset.csv"
QUALITY_GRADE = "research"
PER_PAGE      = 50
DELAY_S       = 0.01
PHOTO_LICENSE = ["cc-by", "cc-by-nc", "cc-by-sa", "cc-by-nc-sa", "cc0"]

# Annotations iNaturalist — Partie de plante
# term_id=12  → "Plant Part"
# term_value_id=13 → "Leaf"   (feuille)  ← priorité
# term_value_id=14 → "Flower" (fleur)    ← écarté en priorité 1
# term_value_id=15 → "Fruit"  (fruit)    ← écarté en priorité 1
LEAF_TERM_ID  = 12
LEAF_VALUE_ID = 13

INATURALIST_API = "https://api.inaturalist.org/v1"
HEADERS = {"Accept": "application/json", "User-Agent": "aromate-dataset-builder/1.0"}

# ─────────────────────────────────────────────
#  30 AROMATES — (nom_scientifique, nom_commun_fr)
# ─────────────────────────────────────────────
AROMATES: list[tuple[str, str]] = [
    ("Ocimum basilicum",          "basilic commun"),
    ("Thymus vulgaris",           "thym commun"),
    ("Rosmarinus officinalis",    "romarin"),
    ("Lavandula angustifolia",    "lavande vraie"),
    ("Mentha piperita",           "menthe poivree"),
    ("Mentha spicata",            "menthe verte"),
    ("Origanum vulgare",          "origan"),
    ("Salvia officinalis",        "sauge officinale"),
    ("Coriandrum sativum",        "coriandre"),
    ("Anethum graveolens",        "aneth"),
    ("Foeniculum vulgare",        "fenouil"),
    ("Petroselinum crispum",      "persil"),
    ("Anthriscus cerefolium",     "cerfeuil"),
    ("Allium schoenoprasum",      "ciboulette"),
    ("Artemisia dracunculus",     "estragon"),
    ("Laurus nobilis",            "laurier sauce"),
    ("Melissa officinalis",       "melisse"),
    ("Satureja hortensis",        "sarriette des jardins"),
    ("Levisticum officinale",     "liveche"),
    ("Myrrhis odorata",           "cerfeuil musque"),
    ("Trigonella foenum-graecum", "fenugrec"),
    ("Carum carvi",               "carvi"),
    ("Pimpinella anisum",         "anis vert"),
    ("Cuminum cyminum",           "cumin"),
    ("Hyssopus officinalis",      "hysope officinale"),
    ("Agastache foeniculum",      "agastache fenouil"),
    ("Verbena officinalis",       "verveine officinale"),
    ("Borago officinalis",        "bourrache"),
    ("Calendula officinalis",     "souci officinal"),
    ("Chamaemelum nobile",        "camomille romaine"),
]


# ─────────────────────────────────────────────
#  FONCTIONS
# ─────────────────────────────────────────────

def slug(name: str) -> str:
    """Nom commun → identifiant de dossier sûr (sans accents ni espaces)."""
    replacements = {
        "é": "e", "è": "e", "ê": "e", "ë": "e",
        "à": "a", "â": "a", "ä": "a",
        "î": "i", "ï": "i",
        "ô": "o", "ö": "o",
        "ù": "u", "û": "u", "ü": "u",
        "ç": "c", " ": "_", "/": "-",
    }
    result = name.lower()
    for src, dst in replacements.items():
        result = result.replace(src, dst)
    return result


def fetch_observations(
    taxon_name: str,
    count: int,
    leaf_only: bool = False,
) -> list[dict]:
    """
    Récupère des observations avec photos.
    Si leaf_only=True, filtre sur l'annotation 'Leaf' (term_id=12, term_value_id=13).
    """
    observations = []
    seen_ids: set[int] = set()
    page = 1

    while len(observations) < count:
        params: dict = {
            "taxon_name":    taxon_name,
            "quality_grade": QUALITY_GRADE,
            "photos":        "true",
            "photo_license": ",".join(PHOTO_LICENSE),
            "order":         "votes",
            "order_by":      "votes",
            "per_page":      min(PER_PAGE, count - len(observations)),
            "page":          page,
            "fields":        "id,taxon,photos",
        }
        if leaf_only:
            params["term_id"]       = LEAF_TERM_ID
            params["term_value_id"] = LEAF_VALUE_ID

        try:
            resp = requests.get(
                f"{INATURALIST_API}/observations",
                params=params, headers=HEADERS, timeout=15,
            )
            resp.raise_for_status()
        except requests.RequestException as e:
            print(f"    [ERREUR API] {e}")
            break

        data    = resp.json()
        results = data.get("results", [])
        if not results:
            break

        for obs in results:
            photos = obs.get("photos", [])
            if photos and obs["id"] not in seen_ids:
                url = photos[0].get("url", "").replace("/square.", "/large.")
                if url:
                    observations.append({"obs_id": obs["id"], "url": url})
                    seen_ids.add(obs["id"])

        total = data.get("total_results", 0)
        if len(results) < PER_PAGE or len(observations) >= total:
            break
        page += 1

    return observations[:count]


def download_and_resize(url: str, dest_path: str, size: tuple[int, int]) -> bool:
    try:
        resp = requests.get(url, timeout=15, headers=HEADERS)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        img = img.resize(size, Image.LANCZOS)
        img.save(dest_path, "JPEG", quality=90)
        return True
    except Exception as exc:
        print(f"    [ERREUR DL] {exc}")
        return False


def process_aromate(
    taxon_name: str,
    common_name: str,
    csv_writer,
    index: int,
    total: int,
) -> tuple[int, int]:
    folder_slug = slug(common_name)
    img_dir     = os.path.join(OUTPUT_DIR, folder_slug)
    os.makedirs(img_dir, exist_ok=True)

    print(f"\n{'═'*60}")
    print(f"[{index}/{total}] {common_name}  ({taxon_name})")
    print(f"{'─'*60}")

    # ── Étape 1 : photos annotées 'Leaf' ──────────────────────────
    print("  Recherche photos de FEUILLES (annotation iNaturalist)…")
    observations = fetch_observations(taxon_name, TARGET_COUNT, leaf_only=True)
    leaf_count   = len(observations)
    print(f"  → {leaf_count} photos de feuilles trouvées.")

    # ── Étape 2 : compléter avec des photos générales si besoin ───
    if leaf_count < TARGET_COUNT:
        missing = TARGET_COUNT - leaf_count
        print(f"  Quota incomplet ({missing} manquants) — fallback photos générales…")
        extra = fetch_observations(taxon_name, TARGET_COUNT, leaf_only=False)

        existing_ids = {o["obs_id"] for o in observations}
        for obs in extra:
            if obs["obs_id"] not in existing_ids and len(observations) < TARGET_COUNT:
                observations.append(obs)
                existing_ids.add(obs["obs_id"])

        print(f"  → {len(observations)} photos au total après fallback.")

    if not observations:
        print("  Aucune observation trouvée — aromate ignoré.")
        return 0, 0

    # ── Téléchargement ─────────────────────────────────────────────
    success = 0
    for i, obs in enumerate(observations, start=1):
        filename  = f"{folder_slug}_{i:04d}.jpg"
        dest_path = os.path.join(img_dir, filename)
        rel_path  = os.path.join(OUTPUT_DIR, folder_slug, filename)

        label = "(feuille)" if i <= leaf_count else "(général)"
        print(f"    [{i:>3}/{len(observations)}] {filename} {label} … ", end="", flush=True)

        if download_and_resize(obs["url"], dest_path, IMAGE_SIZE):
            csv_writer.writerow({"nom_aromate": common_name, "chemin_image": rel_path})
            success += 1
            print("OK")
        else:
            print("ECHEC")

        time.sleep(DELAY_S)

    return success, len(observations)


# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, CSV_FILENAME)

    total_ok    = 0
    total_tried = 0
    start_time  = time.time()

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["nom_aromate", "chemin_image"])
        writer.writeheader()

        for idx, (taxon, common) in enumerate(AROMATES, start=1):
            ok, tried = process_aromate(taxon, common, writer, idx, len(AROMATES))
            total_ok    += ok
            total_tried += tried
            f.flush()

    elapsed = int(time.time() - start_time)
    h, m, s = elapsed // 3600, (elapsed % 3600) // 60, elapsed % 60

    print(f"\n{'═'*60}")
    print(f"TERMINÉ — {total_ok}/{total_tried} images téléchargées")
    print(f"Durée totale : {h:02d}h{m:02d}m{s:02d}s")
    print(f"CSV          : {csv_path}")
    print(f"Dossier      : {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

In [ ]:
%pip install ddgs 

In [ ]:
"""
Scraping d'images d'aromates via DuckDuckGo Images
====================================================
Utilise le package `ddgs` (successeur de `duckduckgo_search`)

Installation :
    pip install ddgs Pillow requests

Arborescence produite :
    images/
      basilic_commun/
        basilic_commun_0001.jpg
        …
      dataset.csv
"""

import os
import csv
import time
import requests
from PIL import Image
from io import BytesIO
from ddgs import DDGS

# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────
TARGET_COUNT = 50
IMAGE_SIZE   = (400, 400)
OUTPUT_DIR   = "dataset_aromatic_duckduckgo"
CSV_FILENAME = "dataset.csv"
DELAY_S      = 0.1        # pause entre téléchargements
TIMEOUT_S    = 10

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

# ─────────────────────────────────────────────
#  30 AROMATES
#  Format : (nom_commun_fr, [requêtes_ddg])
# ─────────────────────────────────────────────
AROMATES: list[tuple[str, list[str]]] = [
    ("basilic commun",          ["basilic", "ocimum basilicum"]),
    ("thym commun",             ["thym", "thymus vulgaris"]),
    ("romarin",                 ["romarin", "rosmarinus officinalis"]),
    ("lavande vraie",           ["lavandula angustifolia"]),
    ("menthe poivree",          ["menthe", "mentha piperita"]),
    ("menthe verte",            ["menthe", "mentha spicata"]),
    ("origan",                  ["origan", "origanum vulgare"]),
    ("sauge officinale",        ["sauge", "salvia officinalis"]),
    ("coriandre",               ["coriandre", "coriandrum sativum"]),
    ("aneth",                   ["aneth", "anethum graveolens"]),
    ("fenouil",                 ["fenouil", "foeniculum vulgare"]),
    ("persil",                  ["persil", "petroselinum crispum"]),
    ("cerfeuil",                ["cerfeuil", "anthriscus cerefolium"]),
    ("ciboulette",              ["ciboulette", "allium schoenoprasum"]),
    ("estragon",                ["estragon", "artemisia dracunculus"]),
    ("laurier sauce",           ["laurier sauce", "laurus nobilis"]),
    ("melisse",                 ["melisse", "melissa officinalis"]),
    ("liveche",                 ["liveche", "flevisticum officinale"]),
    ("cerfeuil musque",         ["cerfeuil musque", "sweet cicely"]),
    ("fenugrec",                ["fenugrec", "trigonella foenum-graecum"]),
    ("carvi",                   ["carvi", "carum carvi"]),
    ("anis vert",               ["anis vert", "pimpinella anisum"]),
    ("cumin",                   ["cumin", "cuminum cyminum"]),
    ("hysope officinale",       ["hysope officinale", "hyssopus officinalis"]),
    ("agastache fenouil",       ["agastache fenouil", "agastache foeniculum"]),
    ("verveine officinale",     ["verveine officinale", "verbena officinalis"]),
    ("bourrache",               ["bourrache", "borago officinalis"]),
    ("souci officinal",         ["souci officinal", "calendula officinalis"]),
    ("camomille romaine",       ["camomille romaine", "chamaemelum nobile"]),
]


# ─────────────────────────────────────────────
#  FONCTIONS
# ─────────────────────────────────────────────

def slug(name: str) -> str:
    replacements = {
        "é": "e", "è": "e", "ê": "e", "ë": "e",
        "à": "a", "â": "a", "ä": "a",
        "î": "i", "ï": "i",
        "ô": "o", "ö": "o",
        "ù": "u", "û": "u", "ü": "u",
        "ç": "c", " ": "_", "/": "-",
    }
    result = name.lower()
    for src, dst in replacements.items():
        result = result.replace(src, dst)
    return result


def fetch_image_urls(queries: list[str], count: int) -> list[str]:
    """
    Interroge DuckDuckGo Images via le package `ddgs`.
    API : ddgs.images(query, size=, type_image=, max_results=)
    """
    urls: list[str] = []
    seen: set[str]  = set()

    ddgs = DDGS()

    for query in queries:
        if len(urls) >= count:
            break

        needed = count - len(urls)
        try:
            results = ddgs.images(
                query,                              # ← positional, PAS keywords=
                size="Large",
                type_image="photo",
                max_results=min(needed + 30, 200),  # marge pour les échecs DL
            )
            for r in results:
                url = r.get("image", "")
                if url and url not in seen:
                    urls.append(url)
                    seen.add(url)
                if len(urls) >= count:
                    break

        except Exception as e:
            print(f"    [ERREUR DDG] requête '{query}' → {e}")

        time.sleep(1.0)  # pause entre requêtes DDG pour éviter le rate-limit

    return urls[:count]


def download_and_resize(url: str, dest_path: str, size: tuple[int, int]) -> bool:
    """Télécharge, crop centré et redimensionne en JPEG 400x400."""
    try:
        resp = requests.get(url, timeout=TIMEOUT_S, headers=HEADERS)
        resp.raise_for_status()

        img = Image.open(BytesIO(resp.content)).convert("RGB")

        w, h   = img.size
        target = size[0] / size[1]
        ratio  = w / h

        if ratio > target:
            new_w = int(h * target)
            left  = (w - new_w) // 2
            img   = img.crop((left, 0, left + new_w, h))
        elif ratio < target:
            new_h = int(w / target)
            top   = (h - new_h) // 2
            img   = img.crop((0, top, w, top + new_h))

        img = img.resize(size, Image.LANCZOS)
        img.save(dest_path, "JPEG", quality=90)
        return True

    except Exception as exc:
        print(f"    [ERREUR DL] {str(exc)[:80]}")
        return False


def process_aromate(
    common_name: str,
    queries: list[str],
    csv_writer,
    index: int,
    total: int,
) -> tuple[int, int]:
    folder_slug = slug(common_name)
    img_dir     = os.path.join(OUTPUT_DIR, folder_slug)
    os.makedirs(img_dir, exist_ok=True)

    print(f"\n{'═'*60}")
    print(f"[{index}/{total}] {common_name}")
    print(f"  Requêtes : {queries[0]} (+{len(queries)-1} fallbacks)")
    print(f"{'─'*60}")

    print("  Récupération des URLs…")
    urls = fetch_image_urls(queries, TARGET_COUNT)
    print(f"  → {len(urls)} URLs trouvées.")

    if not urls:
        print("  Aucune URL — aromate ignoré.")
        return 0, 0

    success = 0
    for i, url in enumerate(urls, start=1):
        filename  = f"{folder_slug}_{i:04d}.jpg"
        dest_path = os.path.join(img_dir, filename)
        rel_path  = os.path.join(OUTPUT_DIR, folder_slug, filename)

        print(f"    [{i:>3}/{len(urls)}] {filename} … ", end="", flush=True)

        if download_and_resize(url, dest_path, IMAGE_SIZE):
            csv_writer.writerow({"nom_aromate": common_name, "chemin_image": rel_path})
            success += 1
            print("OK")
        else:
            print("ECHEC")

        time.sleep(DELAY_S)

    return success, len(urls)


# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, CSV_FILENAME)

    total_ok    = 0
    total_tried = 0
    start_time  = time.time()

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["nom_aromate", "chemin_image"])
        writer.writeheader()

        for idx, (common, queries) in enumerate(AROMATES, start=1):
            ok, tried = process_aromate(common, queries, writer, idx, len(AROMATES))
            total_ok    += ok
            total_tried += tried
            f.flush()

    elapsed = int(time.time() - start_time)
    h, m, s = elapsed // 3600, (elapsed % 3600) // 60, elapsed % 60

    print(f"\n{'═'*60}")
    print(f"TERMINÉ — {total_ok}/{total_tried} images téléchargées")
    print(f"Durée totale : {h:02d}h{m:02d}m{s:02d}s")
    print(f"CSV          : {csv_path}")
    print(f"Dossier      : {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()